In [1]:
import os
import re
from pypdf import PdfReader
import docx
from docx.oxml.ns import qn
from docx.table import Table
from docx.text.paragraph import Paragraph

In [2]:
def read_text_file(file_path):
    with open(file_path, 'r',encoding='utf-8') as f:
        text = f.read()
    return text


In [3]:
def read_pdf_file(file_path):
    reader = PdfReader(file_path)
    all_text = []
    for page in reader.pages:
        page_text = page.extract_text()
        all_text.append(page_text)
    return "".join(all_text)
    # return all_text
    

In [4]:
def read_dox(file_path):
    doc = docx.Document(file_path)
    parts = []
    # Header extraction
    for section in doc.sections:
        header_text = "\n".join(p.text for p in section.header.paragraphs if p.text.strip())
        if header_text:
            parts.append(header_text)
    # Main body extraction
    for child in doc.element.body.iterchildren():
        if child.tag == qn("w:p"):
            paragraph = Paragraph(child,doc)
            if paragraph.text.strip():
                parts.append(paragraph.text)
        elif child.tag == qn("w:tbl"):
            table = Table(child,doc)
            for row in table.rows:
                cells_text = [cell.text.strip() for cell in row.cells]
                row_text = "|".join(t for t in cells_text if t)
                if row_text:
                    parts.append(row_text)
    
    # Footer extraction
    for section in doc.sections:
        footer_text = "\n".join(p.text for p in section.footer.paragraphs if p.text.strip())
        if footer_text:
            parts.append(footer_text)
    return "\n\n".join(parts)

In [ ]:
doc_text = read_dox(r"C:\Users\Administrator\OneDrive\Documents\My CV\DevOps\AI generated\Mohammed_Ewees.docx")
print(doc_text)

In [5]:
def read_document(file_path):
    extension = os.path.splitext(file_path)[1].lower()
    if extension == ".txt":
        text = read_text_file(file_path)
    elif extension == ".pdf":
        text = read_pdf_file(file_path)
    elif extension == ".docx":
        text = read_dox(file_path)
    else:
        raise ValueError(f"Unsupported file type: {extension}")
    # clean up messy whitespace so chunking works on tidy text
    text = re.sub(r"[ \t]+"," ", text)   # Collapse repeated spaces/tabs
    text = re.sub(r"\n{3,}","\n\n", text)  # collapse 3+ blank lines to 1
    return text.strip()


In [ ]:
final_text_docx = read_document(r"/home/mohamed-ewees/Downloads/My CV-20260716T121707Z-1-001/My CV/DevOps/AI generated/Mohammed_Ewees.docx")
final_text_pdf = read_document(r"/home/mohamed-ewees/Downloads/My CV-20260716T121707Z-1-001/My CV/DevOps/AI generated/Mohammed_Ewees.pdf")

print("Docx: " , final_text_docx)
print("PDF: " , final_text_pdf)

In [ ]:
# def chunk_text(text, chunk_size = 200, overlap = 50, respect_sentences= True, search_window=100):
#     start = 0
#     chunks = []
#     while start < len(text):
#         ideal_end = start + chunk_size
#         end = ideal_end
#         if respect_sentences and ideal_end < len(text):
#             search_start = max(0,ideal_end-search_window)
#             window = text[search_start:ideal_end]

#             '''Before the for loop runs, we don't yet know whether any sentence ending exists in the window. We need some way to say "nothing found yet" that can't be confused with a real result.
#             that's why we put last_sentence_end=-1'''
#             last_sentence_end = -1
#             for match in re.finditer(r"[.!?](\s|$)", window):
#                 last_sentence_end = match.end()
            
#             if last_sentence_end != -1:
#                 candidate_end = search_start + last_sentence_end
#                 if candidate_end > start:
#                     end = candidate_end
#         end = min(end , len(text))
#         chunk = text[start:end].strip()
#         if chunk:
#             chunks.append(chunk)
#         if end >= len(text):
#             break

#         start = end - overlap
#     return chunks

In [17]:
def chunk_text(
    text,
    chunk_size=500,
    overlap=75,
    respect_sentences=True,
    search_window=100,
):
    """
    Character-based chunking with smart sentence and word boundaries.

    Priority:
    1. Paragraphs
    2. Sentence boundaries
    3. Word boundaries
    4. Hard cut (last resort)
    """

    chunks = []
    start = 0
    text_length = len(text)

    while start < text_length:

        ideal_end = min(start + chunk_size, text_length)
        end = ideal_end

        # ----------------------------------------------------
        # Try to end at a sentence boundary
        # ----------------------------------------------------
        if respect_sentences and ideal_end < text_length:

            # ---------- Look FORWARD first ----------
            forward_limit = min(text_length, ideal_end + search_window)
            forward_text = text[ideal_end:forward_limit]

            forward_match = re.search(r"[.!?](\s|$)", forward_text)

            if forward_match:
                end = ideal_end + forward_match.end()

            else:
                # ---------- Otherwise look BACKWARD ----------
                backward_start = max(start, ideal_end - search_window)
                backward_text = text[backward_start:ideal_end]

                matches = list(re.finditer(r"[.!?](\s|$)", backward_text))

                if matches:
                    end = backward_start + matches[-1].end()

        # ----------------------------------------------------
        # Don't end inside a word
        # ----------------------------------------------------
        if end < text_length and not text[end].isspace():

            while end > start and not text[end - 1].isspace():
                end -= 1

        # Last resort
        if end <= start:
            end = ideal_end

        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        if end >= text_length:
            break

        # ----------------------------------------------------
        # Overlap
        # ----------------------------------------------------
        start = max(0, end - overlap)

        # Don't start inside a word
        while start < text_length and not text[start].isspace():
            start += 1

        # Skip extra whitespace
        while start < text_length and text[start].isspace():
            start += 1

    return chunks

In [18]:
def load_and_chunk(file_path , chunk_size = 200 , overlap = 50 , respect_sentences = True):
    text = read_document(file_path)
    chunks = chunk_text(text , chunk_size=chunk_size , overlap=overlap , respect_sentences=respect_sentences)
    return chunks

In [19]:
if __name__ == "__main__":
    file_path = r"/home/mohamed-ewees/Downloads/My CV-20260716T121707Z-1-001/My CV/DevOps/AI generated/Mohammed_Ewees.docx"
    chunks = load_and_chunk(file_path, chunk_size=200 , overlap=50)
    print(f"Loaded and chunked {file_path} into {len(chunks)} chunks:\n")

    for i, chunk in enumerate(chunks):
        print(f"--- Chunk {i} ({len(chunk)} chars) ---")
        print(chunk)
        print()

Loaded and chunked /home/mohamed-ewees/Downloads/My CV-20260716T121707Z-1-001/My CV/DevOps/AI generated/Mohammed_Ewees.docx into 28 chunks:

--- Chunk 0 (189 chars) ---
MOHAMED EWEES

Senior DevOps Lead | AWS Cloud Infrastructure & Solutions Architecture

Cairo, Egypt | +201064386700 | engmohamedewees@gmail.com | LinkedIn/Mohamed-Ewees | Open to relocation

--- Chunk 1 (281 chars) ---
| LinkedIn/Mohamed-Ewees | Open to relocation 

PROFESSIONAL SUMMARY

AWS Certified Solutions Architect (Associate) and DevOps Lead with 10+ years of experience architecting, securing, and scaling cloud infrastructure for regulated, high-availability financial and telecom systems.

--- Chunk 2 (260 chars) ---
high-availability financial and telecom systems. Proven record leading DevOps transformation for SEPA-compliant payment platforms, cutting deployment time by 50% and improving SLA adherence by 30% through proactive monitoring and multi-region AWS architecture.

--- Chunk 3 (210 chars) ---
monitoring 